# RAW PIPELINE — World Air Pollution & AQI (2014–2025)
## Mục tiêu:
### 1) Download toàn bộ dataset từ Kaggle vào thư mục data/raw
### 2) Đọc FULL dữ liệu (không lọc bớt)
### 3) Tạo datalake theo dạng Parquet, partition theo city/year

In [1]:
!pip install kaggle pandas pyarrow

Defaulting to user installation because normal site-packages is not writeable


In [2]:
pip install kaggle

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import subprocess
import glob
from pathlib import Path

import pandas as pd

### Cell 3 — Download dataset bằng Kaggle CLI → data/raw

In [4]:
# Thư mục gốc project & thư mục raw
PROJECT_ROOT = Path(r"C:\Users\ADMIN\python\Project-Urban-DM")
RAW_DIR      = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Dùng kaggle CLI để download & unzip trực tiếp vào data/raw
cmd = [
    "kaggle", "datasets", "download",
    "-d", "ashyou09/world-air-pollution-and-aqi-dataset-20142025",
    "-p", str(RAW_DIR),
    "--unzip"
]
print("▶️ Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("\n✅ Download xong. Các file trong data/raw:")
for f in RAW_DIR.iterdir():
    try:
        size_mb = f.stat().st_size / 1e6
        print(f"  - {f.name:40s} {size_mb:6.1f} MB")
    except Exception:
        print(f"  - {f.name}")

▶️ Running: kaggle datasets download -d ashyou09/world-air-pollution-and-aqi-dataset-20142025 -p C:\Users\ADMIN\python\Project-Urban-DM\data\raw --unzip

✅ Download xong. Các file trong data/raw:
  - australia                                   0.0 MB
  - bangladesh                                  0.0 MB
  - brazil                                      0.0 MB
  - china                                       0.0 MB
  - egypt                                       0.0 MB
  - ethiopia                                    0.0 MB
  - france                                      0.0 MB
  - germany                                     0.0 MB
  - global_air_quality_2014_2025.csv           71.9 MB
  - india                                       0.0 MB
  - indonesia                                   0.0 MB
  - iran                                        0.0 MB
  - japan                                       0.0 MB
  - mexico                                      0.0 MB
  - nigeria                       

### Cell 4 — Load FULL dữ liệu global vào DataFrame

In [5]:
from pathlib import Path
import pandas as pd
import glob

RAW_DIR = Path(r"C:\Users\ADMIN\python\Project-Urban-DM\data\raw")

# Lấy tất cả file CSV trong raw (gồm global, country folders, v.v.)
csv_files = glob.glob(str(RAW_DIR / "**" / "*.csv"), recursive=True)
print(f"Tổng số file CSV tìm được: {len(csv_files)}")

dfs = []
for file in csv_files:
    try:
        print("Đang đọc:", file)
        tmp = pd.read_csv(file, low_memory=False)
        dfs.append(tmp)
    except Exception as e:
        print(f"Lỗi khi đọc file {file}: {e}")

df = pd.concat(dfs, ignore_index=True)
print(f"\nShape sau khi concat ALL CSV: {df.shape}")
print(f"Tổng số bản ghi: {len(df):,} (~{len(df)/1000:.1f}k)")

Tổng số file CSV tìm được: 52
Đang đọc: C:\Users\ADMIN\python\Project-Urban-DM\data\raw\global_air_quality_2014_2025.csv
Đang đọc: C:\Users\ADMIN\python\Project-Urban-DM\data\raw\australia\australia\australia_air_quality_2014_2025.csv
Đang đọc: C:\Users\ADMIN\python\Project-Urban-DM\data\raw\bangladesh\bangladesh\bangladesh_air_quality_2014_2025.csv
Đang đọc: C:\Users\ADMIN\python\Project-Urban-DM\data\raw\brazil\brazil\brazil_air_quality_2014_2025.csv
Đang đọc: C:\Users\ADMIN\python\Project-Urban-DM\data\raw\china\china\china_air_quality_2014_2025.csv
Đang đọc: C:\Users\ADMIN\python\Project-Urban-DM\data\raw\egypt\egypt\egypt_air_quality_2014_2025.csv
Đang đọc: C:\Users\ADMIN\python\Project-Urban-DM\data\raw\ethiopia\ethiopia\ethiopia_air_quality_2014_2025.csv
Đang đọc: C:\Users\ADMIN\python\Project-Urban-DM\data\raw\france\france\france_air_quality_2014_2025.csv
Đang đọc: C:\Users\ADMIN\python\Project-Urban-DM\data\raw\germany\germany\germany_air_quality_2014_2025.csv
Đang đọc: C:\Us

### Cell 5 — Đếm số dòng raw & kiểm tra missing

In [6]:
n_rows = len(df)          # tương đương df.shape[0]
print(f"Tổng số bản ghi trong raw dataframe: {n_rows:,}")
print(f"Tương đương ~{n_rows/1000:.1f}k records")

print("\nKiểu dữ liệu các cột:")
print(df.dtypes)

print("\nMissing values (chỉ hiện cột có missing):")
missing = df.isnull().sum()
print(missing[missing > 0])

Tổng số bản ghi trong raw dataframe: 1,349,280
Tương đương ~1349.3k records

Kiểu dữ liệu các cột:
Country                         object
State                           object
City                            object
Date                            object
PM2.5 (ug/m3)                  float64
PM10 (ug/m3)                   float64
NO (ug/m3)                     float64
NO2 (ug/m3)                    float64
NOx (ppb)                      float64
NH3 (ug/m3)                    float64
CO (mg/m3)                     float64
SO2 (ug/m3)                    float64
O3 (ug/m3)                     float64
Benzene (ug/m3)                float64
Toluene (ug/m3)                float64
Xylene (ug/m3)                 float64
AQI                            float64
AQI_Bucket                      object
Wind_Speed (km/h)              float64
Humidity (%)                   float64
Deforestation_Rate_%           float64
Industry_Growth_%              float64
CO2_Emission_MT                float64
Popu

### Cell 6 — Xác định cột date & city

In [7]:
# Tìm các cột liên quan tới thời gian và city để xử lý tiếp
date_cols = [c for c in df.columns if any(k in c.lower() for k in ['date', 'time', 'year'])]
city_cols = [c for c in df.columns if 'city' in c.lower()]

print("Date-related columns:", date_cols)
print("City-related columns:", city_cols)

# In thử vài giá trị để chắc chắn
if date_cols:
    print("\nSample date values:")
    print(df[date_cols].head(5))

if city_cols:
    print("\nSample city values:")
    print(df[city_cols[0]].dropna().unique()[:10])

Date-related columns: ['Date']
City-related columns: ['City']

Sample date values:
         Date
0  2014-01-01
1  2014-01-01
2  2014-01-01
3  2014-01-01
4  2014-01-01

Sample city values:
['Canberra' 'Sydney' 'Newcastle' 'Wollongong' 'Central Coast' 'Albury'
 'Maitland' 'Coffs Harbour' 'Wagga Wagga' 'Tamworth']


### Cell 7 — Parse date → tạo cột year

In [8]:
# Tự động chọn cột date (ví dụ: 'Date', 'date', 'datetime', ...)
date_col = next((c for c in df.columns if 'date' in c.lower()), None)
print("Dùng cột date:", date_col)

df[date_col] = pd.to_datetime(df[date_col], errors='coerce')  # parse, lỗi → NaT
df['year']   = df[date_col].dt.year                           # tạo cột year

print("\nPhân bố số bản ghi theo year:")
print(df['year'].value_counts().sort_index())

Dùng cột date: Date

Phân bố số bản ghi theo year:
year
2014    132240
2015    110640
2016    110640
2017    110640
2018    110640
2019    110640
2020    110640
2021    110640
2022    110640
2023    110640
2024    110640
2025    110640
Name: count, dtype: int64


### Cell 8 — Chuẩn hoá city → city_partition

In [9]:
# Tự động chọn cột city (ví dụ: 'City', 'city_name', ...)
city_col = next((c for c in df.columns if 'city' in c.lower()), None)
print("Dùng cột city:", city_col)

# Chuẩn hoá city thành dạng dùng được cho tên folder (lowercase, snake_case)
df['city_partition'] = (
    df[city_col]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', '_', regex=True)    # 'Ho Chi Minh City' → 'ho_chi_minh_city'
    .str.replace(r'[^\w]', '_', regex=True)  # ký tự đặc biệt → '_'
)

print("\nSample city_partition:", df['city_partition'].unique()[:10])
print("Total unique cities:", df['city_partition'].nunique())

Dùng cột city: City

Sample city_partition: ['canberra' 'sydney' 'newcastle' 'wollongong' 'central_coast' 'albury'
 'maitland' 'coffs_harbour' 'wagga_wagga' 'tamworth']
Total unique cities: 2271


### Cell 9 — Build Datalake (Parquet, partition theo city/year)

In [10]:
# Thư mục datalake cho AQI
DATALAKE_DIR = PROJECT_ROOT / "data" / "datalake" / "aqi"

skipped = 0
written = 0

# Group theo city_partition + year rồi ghi từng partition
for (city, year), group in df.groupby(['city_partition', 'year']):
    if pd.isna(year):            # bỏ record không có year (rất ít)
        skipped += len(group)
        continue

    partition_dir = DATALAKE_DIR / str(city) / str(int(year))
    partition_dir.mkdir(parents=True, exist_ok=True)

    out_file = partition_dir / "data.parquet"
    # Bỏ cột tạm city_partition trước khi ghi
    group.drop(columns=['city_partition']).to_parquet(
        out_file,
        index=False,
        engine='pyarrow'
    )
    written += len(group)

print("✅ Datalake đã build xong!")
print(f"   Records đã ghi : {written:,}")
print(f"   Records bỏ qua : {skipped:,} (thiếu year)")
print(f"   Lưu tại        : {DATALAKE_DIR}")

✅ Datalake đã build xong!
   Records đã ghi : 1,349,280
   Records bỏ qua : 0 (thiếu year)
   Lưu tại        : C:\Users\ADMIN\python\Project-Urban-DM\data\datalake\aqi


### Cell 10 — Verify datalake & đếm lại tổng số records

In [11]:
# Đếm số city folder & số file parquet
city_dirs     = [p for p in DATALAKE_DIR.iterdir() if p.is_dir()]
parquet_files = list(DATALAKE_DIR.rglob("*.parquet"))

print(f"Số thành phố (city): {len(city_dirs)}")
print(f"Tổng file parquet  : {len(parquet_files)}")

# Đếm lại tổng số dòng từ toàn bộ parquet (đảm bảo không mất data)
total_rows = 0
for f in parquet_files:
    total_rows += len(pd.read_parquet(f))

print(f"\nTổng số bản ghi trong datalake: {total_rows:,}")
print(f"Tương đương ~{total_rows/1000:.1f}k records")

# Đọc thử 1 file để xem schema
sample_file = parquet_files[0]
sample_df   = pd.read_parquet(sample_file)
print(f"\nSample partition: {sample_file.relative_to(DATALAKE_DIR)}")
print(f"Shape           : {sample_df.shape}")
print(sample_df.dtypes)

Số thành phố (city): 2271
Tổng file parquet  : 27241

Tổng số bản ghi trong datalake: 1,349,280
Tương đương ~1349.3k records

Sample partition: 10th_of_ramadan\2014\data.parquet
Shape           : (48, 30)
Country                                object
State                                  object
City                                   object
Date                           datetime64[ns]
PM2.5 (ug/m3)                         float64
PM10 (ug/m3)                          float64
NO (ug/m3)                            float64
NO2 (ug/m3)                           float64
NOx (ppb)                             float64
NH3 (ug/m3)                           float64
CO (mg/m3)                            float64
SO2 (ug/m3)                           float64
O3 (ug/m3)                            float64
Benzene (ug/m3)                       float64
Toluene (ug/m3)                       float64
Xylene (ug/m3)                        float64
AQI                                   float64
AQI_Bucket   